In [2]:
#oil
from variables_to_specify_oil import *
from training_utilities_2nd_part import *

df, columns_to_normalize, oil_target_col, forecast_avg_target_col_name, avg_target_col_name, No_of_datapoints_in_one_day, start_date, end_date, delta, one_month_days, out_columns, oil_drop_columnss, oil_windows, index_of_one_month, one_month_window_size, date_col_name = variables_to_specify_oil()


from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df[columns_to_normalize] = scaler.fit_transform(df[columns_to_normalize])

df = df.dropna().reset_index(drop=True)

oil_df = df
oil_time_steps = 1
oil_df

component,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
0,2016-07-01 00:00:00,0.615599,0.454943,0.628980,0.467510,0.556576,0.613765,0.691018
1,2016-07-01 01:00:00,0.612708,0.459449,0.626458,0.464878,0.550279,0.620783,0.636233
2,2016-07-01 02:00:00,0.601143,0.436920,0.621438,0.459689,0.512595,0.586144,0.636233
3,2016-07-01 03:00:00,0.599698,0.450437,0.621438,0.462320,0.515693,0.599955,0.581468
4,2016-07-01 04:00:00,0.605480,0.450437,0.626458,0.467510,0.521990,0.599955,0.519656
...,...,...,...,...,...,...,...,...
17395,2018-06-25 19:00:00,0.695081,0.509011,0.737019,0.529859,0.465414,0.558750,0.292132
17396,2018-06-25 20:00:00,0.770227,0.554069,0.790615,0.553249,0.581767,0.593163,0.280891
17397,2018-06-25 21:00:00,0.758662,0.572091,0.793137,0.589577,0.544084,0.551732,0.280891
17398,2018-06-25 22:00:00,0.784682,0.657633,0.828301,0.706454,0.512595,0.586144,0.272466


# stationary

In [3]:
oil_len_of_training_data_of_stationary_model =7*No_of_datapoints_in_one_day

stationary_model = new_copied_lstm_statinary(oil_df, oil_len_of_training_data_of_stationary_model, oil_target_col, oil_drop_columnss, oil_time_steps)

Epoch 1/10


2025-04-09 15:23:05.879923: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - loss: 0.2554
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.1629
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0947
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0443
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0153
Epoch 6/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0070
Epoch 7/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0097
Epoch 8/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0103
Epoch 9/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0079
Epoch 10/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0068
X_train shape: (168, 1, 6)
y_train shape: (168,)
X_train mean: 0.63969636
X_train std: 0.11209777
y_train mean: 0.6183856
y_train std: 0.095594235
y_train min: 0.4269571
y_train max: 0.8497215
total_test_error is : 0.057573143
total_test_error_mae is:  0.20303734
total_time is:  4.756159916
train time is :  0
Model Type: Sequential
Storage Required: 0.08 MB
model stor

# Model reuse

In [4]:
# Model reuse
daily_df_avg = get_elect_daily_avg(oil_df, No_of_datapoints_in_one_day, oil_target_col, avg_target_col_name)


seasonality_periods_acf_ls, seasonality_periods_acf, segmented_daily_df_avg, filtered_most_similar_dict_wass, filtered_most_similar_dict_tvd, forecast_daily_df_avg, segmented_forecast_daily_df_avg, filtered_forecasted_most_similar_dict_wass, filtered_forecasted_most_similar_dict_tvd = get_seasonality_segments_and_similarities(daily_df_avg, avg_target_col_name, forecast_avg_target_col_name, 7)

Detected seasonality periods (ACF): [  7 297 309 314 317 327 330 341 345 351 359]
median_value is:  327


In [5]:
ratio_wass = len(filtered_most_similar_dict_wass)/segmented_daily_df_avg.shape[1]
ratio_tvd = len(filtered_most_similar_dict_tvd)/segmented_daily_df_avg.shape[1]
ratio_forecasted_wass = len(filtered_forecasted_most_similar_dict_wass)/segmented_daily_df_avg.shape[1]
ratio_forecasted_tvd = len(filtered_forecasted_most_similar_dict_tvd)/segmented_daily_df_avg.shape[1]

print('ratio_wass:', ratio_wass)
print('ratio_tvd:', ratio_tvd)
print('ratio_forecasted_wass:', ratio_forecasted_wass)
print('ratio_forecasted_tvd:', ratio_forecasted_tvd)

ratio_wass: 0.7961165048543689
ratio_tvd: 0.941747572815534
ratio_forecasted_wass: 0.8640776699029126
ratio_forecasted_tvd: 0.6796116504854369


## drift detection

In [6]:
df_copy = oil_df[[oil_target_col]]
target_col = oil_target_col
time_steps = oil_time_steps

df_copy['date'] = pd.to_datetime(df_copy.index)
multiplier = No_of_datapoints_in_one_day
x = 7* multiplier
window_len_=[x]

drift_results_df_ls = []
for i in window_len_:
    start_drift_detection_time = timeit.default_timer()
    drift_results_df = detect_drift_univariate(
        df_copy,
        target_col=oil_target_col,
        window_lengths=window_len_,
        arima_order=(1, 0, 0)
    )
    drift_results_df_ls.append(drift_results_df)
    drift_detection_time = timeit.default_timer() - start_drift_detection_time
    num_true = drift_results_df['drift_detected'].sum()
    print("i is: ", i, " and the Number of True values in 'drift_detected':", num_true, " total number of rows are : ", len(drift_results_df))
    print("drift detection time is: ", drift_detection_time)
    drift_results_df = drift_results_df_ls[0]
    drift_indices = list(drift_results_df.index[drift_results_df['drift_detected']])
    print("indices are: ", drift_indices)

Fold 0: Train size=33, Test size=33
Fold 1: Train size=66, Test size=33
Fold 2: Train size=99, Test size=33
Fold 3: Train size=132, Test size=33
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=33, Test size=33
Fold 1: Train size=66, Test size=33
Fold 2: Train size=99, Test size=33
Fold 3: Train size=132, Test size=33
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=33, Test size=33
Fold 1: Train size=66, Test size=33
Fold 2: Train size=99, Test size=33
Fold 3: Train size=132, Test size=33
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=33, Test size=33
Fold 1: Train size=66, Test size=33
Fold 2: Train size=99, Test size=33
Fold 3: Train size=132, Test size=33
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=33, Test size=33
Fold 1: Train size=66, Test size=33
Fold 2: Train size=99, Test size=33
Fold 3: Train size=132, Test size=33
Skipping fold 4: Insufficient training or test data.
Fold 0: Tr

In [7]:
eval_df_monthly2, eval_df_monthly, avg_ml_storage1= new_copied_lstm_reuse_with_hptuning_no_while_loop_with_drift(filtered_most_similar_dict_wass, stationary_model, oil_len_of_training_data_of_stationary_model, oil_df, "SA", oil_target_col, oil_drop_columnss, oil_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices)

print("total reuse reduced count is: ",total_reduced_count_of_retrainings(filtered_most_similar_dict_wass))

window is:  168
Epoch 1/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 111ms/step - loss: 0.2571
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.1641
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0955
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0449
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0158
Epoch 6/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0075
Epoch 7/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 0.0103
Epoch 8/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0111
Epoch 9/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0087
Epoch 10/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0074
i/window is :  1.0
Epoch 1/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.4089
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.2915
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.2026
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.1269
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0647
Epo

In [8]:
eval_df_monthly2, eval_df_monthly, avg_ml_storage2= new_copied_lstm_reuse_with_hptuning_no_while_loop_with_drift(filtered_most_similar_dict_tvd, stationary_model, oil_len_of_training_data_of_stationary_model, oil_df, "SA", oil_target_col, oil_drop_columnss, oil_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices)

print("total reuse reduced count is: ",total_reduced_count_of_retrainings(filtered_most_similar_dict_tvd))

window is:  168
Epoch 1/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - loss: 0.2571
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.1641
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0955
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0449
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0158
Epoch 6/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0075
Epoch 7/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0103
Epoch 8/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0111
Epoch 9/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0087
Epoch 10/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0074
i/window is :  1.0
Epoch 1/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - loss: 0.4089
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.2915
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.2026
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.1269
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0647
Epoc

In [9]:
eval_df_monthly2, eval_df_monthly, avg_ml_storage3= new_copied_lstm_reuse_with_hptuning_no_while_loop_with_drift(filtered_forecasted_most_similar_dict_wass, stationary_model, oil_len_of_training_data_of_stationary_model, oil_df, "ES", oil_target_col, oil_drop_columnss, oil_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices)

print("total reuse reduced count is: ",total_reduced_count_of_retrainings(filtered_forecasted_most_similar_dict_wass))

window is:  168
Epoch 1/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - loss: 0.2571
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.1641
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0955
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0449
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0158
Epoch 6/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0075
Epoch 7/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0103
Epoch 8/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0111
Epoch 9/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0087
Epoch 10/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0074
Epoch 1/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - loss: 0.4089
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.2915
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.2026
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.1269
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0647
Epoch 6/10
6/6 ━━━━━━━━

In [10]:
eval_df_monthly2, eval_df_monthly, avg_ml_storage4= new_copied_lstm_reuse_with_hptuning_no_while_loop_with_drift(filtered_forecasted_most_similar_dict_tvd, stationary_model, oil_len_of_training_data_of_stationary_model, oil_df, "ES", oil_target_col, oil_drop_columnss, oil_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices)

print("total reuse reduced count is: ",total_reduced_count_of_retrainings(filtered_forecasted_most_similar_dict_tvd))

window is:  168
Epoch 1/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 50ms/step - loss: 0.2571
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.1641
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0955
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0449
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0158
Epoch 6/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0075
Epoch 7/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0103
Epoch 8/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0111
Epoch 9/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0087
Epoch 10/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0074
Epoch 1/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - loss: 0.4089
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.2915
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.2026
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.1269
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0647
Epoch 6/10
6/6 ━━━━━━━━

In [11]:
avg_ml_storage_reuse = (avg_ml_storage1+avg_ml_storage2+avg_ml_storage3+avg_ml_storage4)/4
print("avg_ml_storage_reuse is : ", avg_ml_storage_reuse)


avg_ml_storage_reuse is :  0.07739639282226562


# informed retraining

In [12]:
lstm_informed_update(stationary_model,oil_df, oil_target_col, oil_drop_columnss,time_steps, seasonality_periods_acf,No_of_datapoints_in_one_day, drift_indices)

window is:  168
Epoch 1/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 5s 59ms/step - loss: 0.2571
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.1641
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0955
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0449
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0158
Epoch 6/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0075
Epoch 7/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0103
Epoch 8/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0111
Epoch 9/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0087
Epoch 10/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0074
Model Type: Sequential
Storage Required: 0.08 MB
window is:  336
Epoch 1/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 66ms/step - loss: 0.5754
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.4491
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.3459
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.2482
Epoch 5/10
6/6 ━━━━━━━

# periodical

In [7]:
mean_mse_per_window, min_index,mean_mae_per_window = periodical_lstm_training(oil_df, oil_target_col, oil_time_steps, oil_windows, oil_drop_columnss)

window size is :  120
Epoch 1/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 111ms/step - loss: 0.2383
Epoch 2/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.1746
Epoch 3/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.1250
Epoch 4/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0844
Epoch 5/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0516
Epoch 6/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0268
Epoch 7/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0118
Epoch 8/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0058
Epoch 9/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0060
Epoch 10/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0081
Model Type: Sequential
Storage Required: 0.08 MB
Epoch 1/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - loss: 0.5120
Epoch 2/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.4163
Epoch 3/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.3393
Epoch 4/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.2724
Epoch 5/10
4/4 ━━━━━━━━━━━━━━━━